In [1]:
import sys

print(sys.executable)

/workspaces/2D-Optimisation-in-Metals-Manufacturing/.venv/bin/python


In [2]:
import random
from itertools import product
from pathlib import Path

import pandas as pd

from cutting_benchmark.rectangle_datasets import (
    load_datasets,
)

from cutting_benchmark.solvers.rectpack_adapter import (
    ALGORITHMS,
    SORTS,
    run_pack,
)

In [3]:
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / ".git").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root")

RESULTS_DIR = PROJECT_ROOT / "results" / "rectpack"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "pilot_v1.json"
)

datasets = load_datasets(DATASET_PATH)

print(f"Loaded {len(datasets)} datasets")
print(datasets[0]["dataset_id"])

Loaded 30 datasets
rect_v1_mixed_sheet2500x1250_n35_seed0


In [5]:
assert len(datasets) == 30

assert len({
    dataset["dataset_id"]
    for dataset in datasets
}) == len(datasets)

assert all(
    len(dataset["parts"]) == dataset["n_parts"]
    for dataset in datasets
)

In [6]:
CONFIGURATIONS = [
    {
        "algorithm": algorithm,
        "sort": sort,
        "rotation": rotation,
    }
    for algorithm, sort, rotation in product(
        ALGORITHMS,
        SORTS,
        [False, True],
    )
]

assert len(CONFIGURATIONS) == 32

In [7]:
dataset = datasets[0]

test_result = run_pack(
    parts=dataset["parts"],
    sheet_size=dataset["sheet_size"],
    algorithm="MaxRectsBssf",
    sort="area",
    rotation=False,
    return_layout=True,
)

assert test_result["complete"]
assert test_result["parts_packed"] == dataset["n_parts"]
assert test_result["solver"] == "rectpack"

print({
    key: test_result[key]
    for key in [
        "solver",
        "config_id",
        "parts_total",
        "parts_packed",
        "sheets_used",
        "utilisation",
    ]
})

{'solver': 'rectpack', 'config_id': 'MaxRectsBssf__area__BBF', 'parts_total': 35, 'parts_packed': 35, 'sheets_used': 5, 'utilisation': 0.776518016}


In [8]:
def run_benchmark(
    datasets,
    configurations=CONFIGURATIONS,
    repeats=1,
    available_sheets=None,
    execution_seed=123,
):
    rng = random.Random(execution_seed)

    jobs = [
        (dataset, configuration, repeat)
        for dataset in datasets
        for configuration in configurations
        for repeat in range(repeats)
    ]

    rng.shuffle(jobs)

    records = []

    for dataset, configuration, repeat in jobs:
        result = run_pack(
            parts=dataset["parts"],
            sheet_size=dataset["sheet_size"],
            available_sheets=available_sheets,
            **configuration,
        )

        records.append({
            "dataset_id": dataset["dataset_id"],
            "seed": dataset["seed"],
            "profile": dataset["profile"],
            "n_parts": dataset["n_parts"],
            "repeat": repeat,
            **result,
        })

    return pd.DataFrame(records)

In [9]:
results = run_benchmark(
    datasets,
    configurations=CONFIGURATIONS,
    repeats=1,
)

expected_runs = len(datasets) * len(CONFIGURATIONS)

assert len(results) == expected_runs
assert results["complete"].all()

print(f"Completed {len(results)} runs")

Completed 960 runs


In [15]:
results.to_csv(
    RESULTS_DIR / "rectpack_benchmark_refactor_check.csv",
    index=False,
)

In [10]:
config_cols = ["algorithm", "sort", "rotation"]

dataset_config = (
    results
    .groupby(
        ["dataset_id", "profile", "n_parts"] + config_cols,
        as_index=False,
    )
    .agg(
        complete=("complete", "all"),
        sheets_used=("sheets_used", "first"),
        utilisation=("utilisation", "first"),
        area_fraction=("area_fraction", "first"),
        part_fraction=("part_fraction", "first"),
        sheet_gap_to_area_bound=(
            "sheet_gap_to_area_bound", "first"
        ),
        runtime_seconds=("runtime_seconds", "median"),
    )
)

In [11]:
comparison_groups = ["dataset_id", "rotation"]

dataset_config["best_observed_sheets"] = (
    dataset_config
    .groupby(comparison_groups)["sheets_used"]
    .transform("min")
)

dataset_config["extra_sheets_vs_best"] = (
    dataset_config["sheets_used"]
    - dataset_config["best_observed_sheets"]
)

dataset_config["relative_excess"] = (
    dataset_config["sheets_used"]
    / dataset_config["best_observed_sheets"]
    - 1
)

dataset_config["tied_best"] = (
    dataset_config["extra_sheets_vs_best"] == 0
)

In [12]:
summary = (
    dataset_config
    .groupby(config_cols, as_index=False)
    .agg(
        datasets=("dataset_id", "nunique"),
        mean_sheets=("sheets_used", "mean"),
        mean_utilisation=("utilisation", "mean"),
        mean_extra_sheets=("extra_sheets_vs_best", "mean"),
        mean_relative_excess=("relative_excess", "mean"),
        tied_best_rate=("tied_best", "mean"),
        mean_gap_to_area_bound=(
            "sheet_gap_to_area_bound", "mean"
        ),
        median_runtime=("runtime_seconds", "median"),
        p95_runtime=(
            "runtime_seconds",
            lambda x: x.quantile(0.95),
        ),
    )
    .sort_values(
        ["rotation", "mean_relative_excess", "median_runtime"]
    )
)

summary.to_csv(
    RESULTS_DIR / "rectpack_benchmark_summary.csv",
    index=False,
)

formats = {
    "mean_sheets": "{:.2f}",
    "mean_utilisation": "{:.1%}",
    "mean_extra_sheets": "{:.2f}",
    "mean_relative_excess": "{:.1%}",
    "tied_best_rate": "{:.1%}",
    "mean_gap_to_area_bound": "{:.2f}",
    "median_runtime": "{:.4f}",
    "p95_runtime": "{:.4f}",
}

summary_display = summary.copy()

for column, fmt in formats.items():
    summary_display[column] = summary_display[column].map(fmt.format)

display(summary_display)

,algorithm,sort,rotation,datasets,mean_sheets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
20,MaxRectsBssf,long_side,False,30,13.80,68.7%,0.13,1.4%,86.7%,5.03,0.0026,0.0146
12,MaxRectsBaf,long_side,False,30,13.87,68.0%,0.20,2.2%,80.0%,5.10,0.0028,0.0103
4,GuillotineBafSas,long_side,False,30,13.87,67.7%,0.20,3.3%,80.0%,5.10,0.0018,0.0086
16,MaxRectsBssf,area,False,30,13.90,66.7%,0.23,4.6%,80.0%,5.13,0.0035,0.0164
22,MaxRectsBssf,short_side,False,30,13.93,66.5%,0.27,5.0%,73.3%,5.17,0.0024,0.0125
8,MaxRectsBaf,area,False,30,13.90,66.4%,0.23,5.0%,76.7%,5.13,0.0034,0.0107
14,MaxRectsBaf,short_side,False,30,13.93,66.2%,0.27,5.4%,73.3%,5.17,0.0034,0.0083
30,SkylineBl,short_side,False,30,14.33,63.6%,0.67,9.2%,46.7%,5.57,0.0077,0.0276
28,SkylineBl,long_side,False,30,14.53,62.7%,0.87,10.6%,26.7%,5.77,0.0089,0.0429
18,MaxRectsBssf,input_order,False,30,14.57,62.0%,0.90,12.6%,33.3%,5.80,0.0032,0.0102


In [16]:
summary.to_csv(
    RESULTS_DIR / "rectpack_summary_refactor_check.csv",
    index=False,
)

In [13]:
from cutting_benchmark.rectangle_metrics import (
    calculate_rectangle_metrics,
)


metrics = calculate_rectangle_metrics(
    parts=[
        ("A", 4, 5),
        ("B", 6, 5),
    ],
    placements=[
        (0, 0, 0, 4, 5, "A"),
        (0, 4, 0, 6, 5, "B"),
    ],
    used_sheets=[(10, 10)],
    sheet_size=(10, 10),
    rotation=False,
)

assert metrics["complete"]
assert metrics["parts_packed"] == 2
assert metrics["requested_area"] == 50
assert metrics["packed_area"] == 50
assert metrics["utilisation"] == 0.5
assert metrics["area_lower_bound"] == 1
assert metrics["sheet_gap_to_area_bound"] == 0

In [14]:
incomplete_metrics = calculate_rectangle_metrics(
    parts=[
        ("A", 4, 5),
        ("B", 6, 5),
    ],
    placements=[
        (0, 0, 0, 4, 5, "A"),
    ],
    used_sheets=[(10, 10)],
    sheet_size=(10, 10),
    rotation=False,
)

assert not incomplete_metrics["complete"]
assert incomplete_metrics["part_fraction"] == 0.5
assert incomplete_metrics["area_fraction"] == 0.4

assert pd.isna(
    incomplete_metrics["sheet_gap_to_area_bound"]
)

In [17]:
previous_results = pd.read_csv(
    RESULTS_DIR / "rectpack_benchmark_refactor_check.csv"
)

keys = [
    "dataset_id",
    "algorithm",
    "sort",
    "rotation",
    "repeat",
]

columns = sorted(
    column
    for column in previous_results.columns
    if column != "runtime_seconds"
)

previous = (
    previous_results
    .sort_values(keys)
    .reset_index(drop=True)
)

current = (
    results
    .sort_values(keys)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    previous[columns],
    current[columns],
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("All previous non-runtime result columns match.")

All previous non-runtime result columns match.


In [ ]:
from cutting_benchmark.benchmark import run_benchmark

In [ ]:
results = run_benchmark(
    datasets=datasets,
    configurations=CONFIGURATIONS,
    solver=run_pack,
    repeats=1,
)

In [18]:
from cutting_benchmark.summaries import summarise_results

tables = summarise_results(results)

dataset_config = tables["dataset_config"]
completion_summary = tables["completion_summary"]
summary = tables["summary"]
summary_by_workload = tables["summary_by_workload"]

In [19]:
display(completion_summary)
display(summary)

,solver,config_id,rotation,datasets,completion_rate
0,rectpack,GuillotineBafSas__area__BBF,False,30,1.0
1,rectpack,GuillotineBafSas__area__BBF,True,30,1.0
2,rectpack,GuillotineBafSas__input_order__BBF,False,30,1.0
3,rectpack,GuillotineBafSas__input_order__BBF,True,30,1.0
4,rectpack,GuillotineBafSas__long_side__BBF,False,30,1.0
5,rectpack,GuillotineBafSas__long_side__BBF,True,30,1.0
6,rectpack,GuillotineBafSas__short_side__BBF,False,30,1.0
7,rectpack,GuillotineBafSas__short_side__BBF,True,30,1.0
8,rectpack,MaxRectsBaf__area__BBF,False,30,1.0
9,rectpack,MaxRectsBaf__area__BBF,True,30,1.0


,solver,config_id,rotation,datasets,mean_sheets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
0,rectpack,MaxRectsBssf__long_side__BBF,False,30,13.800000,0.686840,0.133333,0.014323,0.866667,5.033333,0.002636,0.014575
1,rectpack,MaxRectsBaf__long_side__BBF,False,30,13.866667,0.680235,0.200000,0.022116,0.800000,5.100000,0.002765,0.010313
2,rectpack,GuillotineBafSas__long_side__BBF,False,30,13.866667,0.676762,0.200000,0.032557,0.800000,5.100000,0.001771,0.008580
3,rectpack,MaxRectsBssf__area__BBF,False,30,13.900000,0.666762,0.233333,0.046140,0.800000,5.133333,0.003544,0.016352
4,rectpack,MaxRectsBssf__short_side__BBF,False,30,13.933333,0.664666,0.266667,0.050014,0.733333,5.166667,0.002398,0.012459
5,rectpack,MaxRectsBaf__area__BBF,False,30,13.900000,0.663741,0.233333,0.050317,0.766667,5.133333,0.003378,0.010742
6,rectpack,MaxRectsBaf__short_side__BBF,False,30,13.933333,0.661818,0.266667,0.053929,0.733333,5.166667,0.003355,0.008282
7,rectpack,SkylineBl__short_side__BBF,False,30,14.333333,0.635546,0.666667,0.091819,0.466667,5.566667,0.007707,0.027583
8,rectpack,SkylineBl__long_side__BBF,False,30,14.533333,0.626885,0.866667,0.105511,0.266667,5.766667,0.008913,0.042859
9,rectpack,MaxRectsBssf__input_order__BBF,False,30,14.566667,0.619890,0.900000,0.125639,0.333333,5.800000,0.003222,0.010166
